In [2]:
!hmmsearch --max --nonull2 profiles/PF30808.hmm sequences/A0A010YIN8.fa

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.4 (Aug 2023); http://hmmer.org/
# Copyright (C) 2023 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  profiles/PF30808.hmm
# target sequence database:        sequences/A0A010YIN8.fa
# Max sensitivity mode:            on [all heuristic filters off]
# null2 bias corrections:          off
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       DUF8746  [M=64]
Accession:   PF30808.1
Description: Domain of unknown function (DUF8746)
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence   Description
    ------- ------ -----    ------- ------ -----   ---- --  --------   -----------
    1.3e-15   43.8   0.0    4.3

In [1]:
import numpy as np
import pandas as pd
import pyhmmer
import math

In [2]:
transition_df = pd.read_csv('transition.csv', index_col=0)
emission_df = pd.read_csv('emission.csv', index_col=0)

# MAKE STOCHASTIC
trans_len = len(transition_df)
transition_df.iloc[trans_len - 1, trans_len - 1] = 1 # loops back to end state
emission_df.loc['$'] = 0 # added symbol $ for end state
emission_df.iloc[20, trans_len - 1] = 1

A = transition_df.to_numpy()
B = emission_df.to_numpy()

In [3]:
def observable_T(A, B, y):
    C = np.diag(B[y])
    return C @ A

symbols = list(emission_df.index)
n = A.shape[0]
s = B.shape[0]
T = {}
for i, s in enumerate(symbols):
    T_i = observable_T(A, B, i)
    T[s] = T_i

In [4]:
with pyhmmer.easel.SequenceFile("sequences/A0A010YIN8.fa", digital=False) as sf:
    sequences = list(sf)

# for seq in sequences:
seq = sequences[0]
x = np.array([[1] if i == 0 else [0] for i in range(len(A))], dtype=np.float128)
for char in seq.sequence:
    sub = T[char] @ x
    x = sub
x = T['$'] @ x

Log-odds Ratio

In [5]:
log2_model = np.log2(x.sum())
log2_null = len(seq.sequence) * np.log2(0.05)
log_odds = log2_model - log2_null
log_odds

np.longdouble('-20.605994235880919607')

In [16]:
def dagger(A):
    return np.conj(A).T

def kraus_operators(A, B):
    n = A.shape[0]
    s = B.shape[0]
    T = {}
    for i in range(s):
        T_i = observable_T(A, B, i)
        for j in range(n):
            sqrt_T_i = np.zeros(T_i.shape)
            sqrt_T_i[:,j] += np.sqrt(T_i[:,j])
            T[j,i] = sqrt_T_i
    return T

n = A.shape[0]
s = B.shape[0]

T = kraus_operators(A,B)

sum_i_j = []
for i in range(n):
    sum_j = []
    for j in range(s):
        if j == 0:
            sum_j = dagger(T[i, j]) @ T[i, j]
        else:
            sum_j += dagger(T[i, j]) @ T[i, j]
    # print(sum_j)
    # print()
    if i == 0:
        sum_i_j = sum_j.copy()
    else:
        sum_i_j += sum_j

print("FINAL")
print(sum_i_j.sum())

FINAL
131.00025497373736
